In [28]:
# TODO exploratory data visualization

import pandas as pd 
import numpy as np 

# use MAF 01 for 22N9 
sel_n9_geno = pd.read_csv("/work/hs325/csgs2026/data/selectedlines/geno/22N9_MAF01_Genotypes_V1.csv")
sel_all_geno = pd.read_csv("/work/hs325/csgs2026/data/selectedlines/geno/SelectedLines_(All)_MAF01_Genotypes_V1.csv")
sel_n9_pheno = pd.read_csv("/work/hs325/csgs2026/data/selectedlines/pheno/22N9_Phenotypes_V1.csv")
sel_all_pheno = pd.read_csv("/work/hs325/csgs2026/data/selectedlines/pheno/SelectedLines_Phenotypes_V1.csv")

# use MAF 05 for wild
wild_22dbw_geno = pd.read_csv("/work/hs325/csgs2026/data/wildlines/geno/Wild_22DBW_MAF05_Genotypes_V1.csv") 
wild_all_geno = pd.read_csv("/work/hs325/csgs2026/data/wildlines/geno/Wild_Both_Corrected_Genotypes_V1.csv") 
wild_22dbw_pheno =pd.read_csv("/work/hs325/csgs2026/data/wildlines/pheno/Wild_(22DBW)_Phenotypes_V1.csv") 
wild_all_pheno = pd.read_csv("/work/hs325/csgs2026/data/wildlines/pheno/Wild_(Both)_Phenotypes_V1.csv") 


In [29]:
# preview dataframes
# sel_n9_geno 
# sel_all_geno
# wild_all_pheno 
wild_all_geno
print(wild_all_pheno)


      Group SampleID Status  Height  Length  Width    WWt  Survival  \
0      21DW     C207   Dead   58.02     NaN    NaN  33.42       NaN   
1      21DW     C710   Dead   53.65     NaN    NaN  20.67       NaN   
2      21DW     C101   Dead   55.33     NaN    NaN  37.61       NaN   
3      21DW     C111   Dead   53.75     NaN    NaN  26.95       NaN   
4      21DW     C117   Dead   50.01     NaN    NaN  26.29       NaN   
...     ...      ...    ...     ...     ...    ...    ...       ...   
1146  22DBW  645_DBW   Live   61.35     NaN    NaN  35.49       NaN   
1147  22DBW  646_DBW   Live   70.28     NaN    NaN  44.23       NaN   
1148  22DBW  647_DBW   Live   60.96     NaN    NaN  39.05       NaN   
1149  22DBW  652_DBW   Live   61.65     NaN    NaN  37.88       NaN   
1150  22DBW  670_DBW   Live   54.27     NaN    NaN  29.34       NaN   

     Date.at.death  Sampling.Date       Birth   T2D  Adj_T2D Adj_Paul  \
0       2023-05-10            NaN  2021-06-07   711      NaN     True   
1

In [30]:
# check shapes
print("sel_n9_geno shape:", sel_n9_geno.shape)
print("sel_all_geno shape:", sel_all_geno.shape)
print("sel_n9_pheno shape:", sel_n9_pheno.shape)
print("sel_all_pheno shape:", sel_all_pheno.shape)
print("wild_22dbw_geno shape:", wild_22dbw_geno.shape)
print("wild_all_geno shape:", wild_all_geno.shape)
print("wild_22dbw_pheno shape:", wild_22dbw_pheno.shape)
print("wild_all_pheno shape:", wild_all_pheno.shape)

sel_n9_geno shape: (975, 56088)
sel_all_geno shape: (2767, 59289)
sel_n9_pheno shape: (956, 17)
sel_all_pheno shape: (2772, 17)
wild_22dbw_geno shape: (535, 44672)
wild_all_geno shape: (1083, 56617)
wild_22dbw_pheno shape: (562, 17)
wild_all_pheno shape: (1151, 17)


In [31]:
data_dict = {
    "sel_n9": {"geno": sel_n9_geno, "pheno": sel_n9_pheno},
    "sel_all": {"geno": sel_all_geno, "pheno": sel_all_pheno},
    "wild_22dbw": {"geno": wild_22dbw_geno, "pheno": wild_22dbw_pheno},
    "wild_all": {"geno": wild_all_geno, "pheno": wild_all_pheno}
}

print("--- Aligning via Columns (No Index Alteration) ---")
for prefix, dfs in data_dict.items():
    geno_df = dfs["geno"].copy()
    pheno_df = dfs["pheno"].copy()
    
    # The genotype file's row names default to the very first column when loaded normally
    geno_id_col = geno_df.columns[0] 
    
    # Rename it to 'ID' explicitly so it matches your ML model requirements perfectly
    geno_df = geno_df.rename(columns={geno_id_col: "ID"})
    geno_id_col = "ID"
    
    # Use 'ID' for the phenotype file as well
    pheno_id_col = "SampleID" if "SampleID" in pheno_df.columns else pheno_df.columns[1]
    
    # Intersect the individual string IDs across both datasets
    common_ids = set(geno_df[geno_id_col]).intersection(set(pheno_df[pheno_id_col]))
    common_ids_sorted = sorted(list(common_ids))
    
    # Filter and sort dataframes based entirely on column strings
    geno_filtered = geno_df[geno_df[geno_id_col].isin(common_ids_sorted)].set_index(geno_id_col).loc[common_ids_sorted].reset_index()
    pheno_filtered = pheno_df[pheno_df[pheno_id_col].isin(common_ids_sorted)].set_index(pheno_id_col).loc[common_ids_sorted].reset_index()
    
    # Save the aligned dataframes back into our collection
    data_dict[prefix]["geno"] = geno_filtered
    data_dict[prefix]["pheno"] = pheno_filtered
    
    print(f"Aligned {prefix: <10} -> New Genotype Shape: {geno_filtered.shape} | New Phenotype Shape: {pheno_filtered.shape}")

# 3. Unpack variables back to the environment
sel_n9_geno_new, sel_n9_pheno_new = data_dict["sel_n9"]["geno"], data_dict["sel_n9"]["pheno"]
sel_all_geno_new, sel_all_pheno_new = data_dict["sel_all"]["geno"], data_dict["sel_all"]["pheno"]
wild_22dbw_geno_new, wild_22dbw_pheno_new = data_dict["wild_22dbw"]["geno"], data_dict["wild_22dbw"]["pheno"]
wild_all_geno_new, wild_all_pheno_new = data_dict["wild_all"]["geno"], data_dict["wild_all"]["pheno"]

--- Aligning via Columns (No Index Alteration) ---
Aligned sel_n9     -> New Genotype Shape: (947, 56088) | New Phenotype Shape: (947, 17)
Aligned sel_all    -> New Genotype Shape: (2739, 59289) | New Phenotype Shape: (2739, 17)
Aligned wild_22dbw -> New Genotype Shape: (535, 44672) | New Phenotype Shape: (535, 17)
Aligned wild_all   -> New Genotype Shape: (1083, 56617) | New Phenotype Shape: (1083, 17)


In [32]:
# --- Save Selected Lines (Genotypes) ---
sel_n9_geno_new.to_csv("/work/hs325/csgs2026/data/selectedlines/geno/sel_n9_geno.csv", index=False)
sel_all_geno_new.to_csv("/work/hs325/csgs2026/data/selectedlines/geno/sel_all_geno.csv", index=False)

# --- Save Selected Lines (Phenotypes) ---
sel_n9_pheno_new.to_csv("/work/hs325/csgs2026/data/selectedlines/pheno/sel_n9_pheno.csv", index=False)
sel_all_pheno_new.to_csv("/work/hs325/csgs2026/data/selectedlines/pheno/sel_all_pheno.csv", index=False)

# --- Save Wild Lines (Genotypes) ---
wild_22dbw_geno_new.to_csv("/work/hs325/csgs2026/data/wildlines/geno/wild_22dbw_geno.csv", index=False)
wild_all_geno_new.to_csv("/work/hs325/csgs2026/data/wildlines/geno/wild_all_geno.csv", index=False)

# --- Save Wild Lines (Phenotypes) ---
wild_22dbw_pheno_new.to_csv("/work/hs325/csgs2026/data/wildlines/pheno/wild_22dbw_pheno.csv", index=False)
wild_all_pheno_new.to_csv("/work/hs325/csgs2026/data/wildlines/pheno/wild_all_pheno.csv", index=False)

### Sandbox for phenotype_gwas

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

# =============================================================================
# CONFIGURE FILES
# =============================================================================

GENO_FILE = "/work/hs325/csgs2026/data/wildlines/geno/wild_22dbw_geno.csv"
PHENO_FILE = "/work/hs325/csgs2026/data/wildlines/pheno/wild_22dbw_pheno.csv"

# =============================================================================
# LOAD DATA
# =============================================================================

geno_df = pd.read_csv(GENO_FILE)
pheno_df = pd.read_csv(PHENO_FILE)

print("Genotype shape:", geno_df.shape)
print("Phenotype shape:", pheno_df.shape)

geno_df = geno_df.set_index("ID")
pheno_df = pheno_df.set_index("SampleID")

common_ids = geno_df.index.intersection(pheno_df.index)

print("Matched IDs:", len(common_ids))

geno_df = geno_df.loc[common_ids]
pheno_df = pheno_df.loc[common_ids]

ax_cols = [c for c in geno_df.columns if c.startswith("AX")]

X = geno_df[ax_cols].values
y = pd.to_numeric(
    pheno_df["phenotype_gwas"],
    errors="coerce"
).values

mask = ~np.isnan(y)

X = X[mask]
y = y[mask]

print("\nFinal matrix dimensions")
print("X:", X.shape)
print("y:", y.shape)

# =============================================================================
# SINGLE FOLD TRAINING TEST
# =============================================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

train_idx, test_idx = next(kf.split(X))

X_train = X[train_idx]
X_test = X[test_idx]

y_train = y[train_idx]
y_test = y[test_idx]

print("\nTraining samples:", len(train_idx))
print("Testing samples :", len(test_idx))

# =============================================================================
# SCALE
# =============================================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =============================================================================
# FIT MODEL
# =============================================================================

model = Ridge(alpha=1.0)

model.fit(X_train, y_train)

preds = model.predict(X_test)

# =============================================================================
# METRICS
# =============================================================================

mse = mean_squared_error(y_test, preds)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

corr = np.corrcoef(y_test, preds)[0, 1]

print("\nModel Performance")
print("------------------------")
print("MSE      :", mse)
print("RMSE     :", rmse)
print("MAE      :", mae)
print("R2       :", r2)
print("PearsonR :", corr)

# =============================================================================
# GEBV OUTPUT EXAMPLE
# =============================================================================

gebv_df = pd.DataFrame({
    "Observed": y_test,
    "Predicted_GEBV": preds
})

display(gebv_df.head())

print("\nPrediction range:")
print("Min:", preds.min())
print("Max:", preds.max())

Genotype shape: (535, 44672)
Phenotype shape: (535, 17)
Matched IDs: 535

Final matrix dimensions
X: (535, 44671)
y: (535,)

Training samples: 428
Testing samples : 107

Model Performance
------------------------
MSE      : 0.9977194019737491
RMSE     : 0.9988590501035415
MAE      : 0.810771245445644
R2       : -0.1323085748496522
PearsonR : 0.19252344994593606


,Observed,Predicted_GEBV
0,0.258188,-0.575356
1,-1.191022,-0.518178
2,-1.191022,-0.473979
3,-1.191022,-0.994265
4,-0.186138,0.173420



Prediction range:
Min: -1.4652602246221826
Max: 1.5754344192407443


### Sandbox

In [33]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from skopt import BayesSearchCV
from skopt.space import Integer
from sklearn.metrics import make_scorer

In [35]:
def pearson_corr_func(y_true, y_pred):
    if y_pred.ndim == 2:  
        y_pred = y_pred[:, 1] 
    if np.allclose(y_pred, y_pred[0]) or len(np.unique(y_true)) < 2:
        return 0.0
    corr, _ = pearsonr(y_pred, y_true)
    return corr if not np.isnan(corr) else 0.0

# Define custom scorer using pearson correlation for BayesSearchCV
pearson_scorer = make_scorer(
    pearson_corr_func,
    response_method="predict_proba"
)

GENO_PATH = "/work/hs325/csgs2026/data/selectedlines/geno/sel_n9_geno.csv"
PHENO_PATH = "/work/hs325/csgs2026/data/selectedlines/pheno/sel_n9_pheno.csv"

print("--- Step 1: Loading Data ---")
geno_df = pd.read_csv(GENO_PATH)
pheno_df = pd.read_csv(PHENO_PATH)

print(f"Loaded Genotype shape: {geno_df.shape} (with 'ID' column)")
print(f"Loaded Phenotype shape: {pheno_df.shape} (with 'SampleID' column)")


--- Step 1: Loading Data ---
Loaded Genotype shape: (947, 56088) (with 'ID' column)
Loaded Phenotype shape: (947, 17) (with 'SampleID' column)


In [36]:
print("\n--- Step 2: Aligning and Extracting Matrices ---")
# Set the respective ID columns as temporary indices for a clean intersection
geno_temp = geno_df.set_index("ID")
pheno_temp = pheno_df.set_index("SampleID")

common_ids = geno_temp.index.intersection(pheno_temp.index)
print(f"Identified {len(common_ids)} common samples.")

# Filter and align using the matching indices
X_df = geno_temp.loc[common_ids]
y_df = pheno_temp.loc[common_ids]

# Isolate genotype feature columns (AX) and phenotype target (status_01)
ax_cols = [col for col in X_df.columns if col.startswith("AX")]
X = X_df[ax_cols].to_numpy()
y = y_df["status_01"].to_numpy()

print(f"Final Matrix X shape: {X.shape}")
print(f"Final Target y shape: {y.shape}")


--- Step 2: Aligning and Extracting Matrices ---
Identified 947 common samples.
Final Matrix X shape: (947, 56087)
Final Target y shape: (947,)


In [39]:
print("\n--- Step 3: Fast Train/Test Split & Scale ---")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n--- Step 4: Rapid Bayesian Optimization ---")
base_model = XGBClassifier(tree_method="hist", eval_metric="logloss")

# Bare-minimum search space for testing
search_space = {
    "n_estimators": Integer(50, 150),
    "max_depth": Integer(3, 6)
}

# Run 3 quick iterations across 3 internal folds
opt = BayesSearchCV(
    estimator=base_model,
    search_spaces=search_space,
    n_iter=3,
    cv=3,
    scoring=pearson_scorer,
    n_jobs=-1,
    random_state=42,
    verbose=0
)

print("Tuning hyperparameters on train split...")
opt.fit(X_train_scaled, y_train)

print(f"Optimal Parameters Discovered: {opt.best_params_}")
print(f"Best In-Fold Pearson R: {opt.best_score_:.4f}")



--- Step 3: Fast Train/Test Split & Scale ---

--- Step 4: Rapid Bayesian Optimization ---
Tuning hyperparameters on train split...
Optimal Parameters Discovered: OrderedDict({'max_depth': 4, 'n_estimators': 123})
Best In-Fold Pearson R: 0.0424


In [40]:

print("\n--- Step 5: Test Best Model Evaluated via Pearson ---")
# Generate predictions on the test set using the best tuned estimator
best_model = opt.best_estimator_
probs = best_model.predict_proba(X_test_scaled)[:, 1]

# Calculate final test metric
final_pearson = pearson_corr_func(y_test, probs)
print(f"Success! Final Test Set Pearson Correlation (r): {final_pearson:.4f}")


--- Step 5: Test Best Model Evaluated via Pearson ---
Success! Final Test Set Pearson Correlation (r): -0.0949
